# 🧶 Multimodal Rug Search — Demo Notebook

This notebook demonstrates the two-part search pipeline:
- **Part 1**: Structured text query → metadata filter + SentenceTransformer ranking
- **Part 2**: Room image (+ optional text) → CLIP embedding + FAISS ANN + fusion scoring

Run in Google Colab or locally after `pip install -r requirements.txt`.

## 0. Setup (Colab only)
If running in Colab, clone the repo and install dependencies first.

In [ ]:
# ── Colab only ───────────────────────────────────────────
# Uncomment the lines below if running in Google Colab
# !git clone https://github.com/<your-repo>/MultiModalSearchAssignment.git
# %cd MultiModalSearchAssignment
# !pip install -r requirements.txt -q
# ─────────────────────────────────────────────────────────

# If running locally, just make sure you are in the project root
import os, sys
# Add project root to path so src.* imports work from the notebook
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('Working dir:', os.getcwd())
print('Python path root:', PROJECT_ROOT)

---
## Part 1 — Structured Text Search

Parses the query with regex + rules, applies soft metadata filters, then ranks with `all-MiniLM-L6-v2`.

In [ ]:
from pathlib import Path
from src.preprocess import build_catalog
from src.query_parser import parse_query

DATA_PATH = Path('..') / 'data' / 'products.csv'
catalog = build_catalog(DATA_PATH)
print(f'Loaded {len(catalog)} products')
print('Sample:', {k: catalog[0][k] for k in ['handle', 'title', 'size', 'price']})

In [ ]:
# Try the query parser
query = '8x10 beige traditional rug'
parsed = parse_query(query)
print(f'Query: "{query}"')
print('Parsed:', parsed)

In [ ]:
import json
from src.search_part1 import search_part1

queries = [
    '8x10 beige traditional rug',
    'runner 2x10 blue rug',
    'small bohemian accent rug',
    '9x12 navy geometric',
    'large neutral rug',
]

for q in queries:
    results = search_part1(q, csv_path=DATA_PATH, top_k=3)
    print(f'\n🔍 "{q}"')
    for r in results:
        print(f"  #{r['rank']} {r['title'][:60]:60s} score={r['final_score']:.3f}")

---
## Part 2 — Multimodal Image + Text Search

Encodes room images + text with CLIP (`openai/clip-vit-base-patch32`), searches a FAISS index of product embeddings, and combines scores with a weighted fusion formula:

```
fusion_score = 0.6 × image_similarity + 0.4 × text_similarity
```

⚠️ First run will download the CLIP model (~600 MB) and take ~30s to build the index.

In [ ]:
from src.search_part2 import search_multimodal

IMAGES_DIR = Path('..') / 'images'
room_image = str(IMAGES_DIR / 'room2.JPG')

# Mode A: Image only
print('Mode A — Image only')
results = search_multimodal(room_image, top_k=5, csv_path=DATA_PATH)
for r in results:
    print(f"  #{r['rank']} {r['title'][:60]:60s} fusion={r['fusion_score']:.3f} img={r['image_similarity']:.3f}")

In [ ]:
# Mode B: Image + text (recommended)
print('Mode B — Image + text query')
results = search_multimodal(
    room_image,
    text_query='warm traditional rug',
    top_k=5,
    csv_path=DATA_PATH,
    include_explanations=True,
)
for r in results:
    print(f"  #{r['rank']} {r['title'][:55]:55s} fusion={r['fusion_score']:.3f}")
    if r.get('explanation'):
        print(f"         💬 {r['explanation']}")

In [ ]:
# Mode C: Text query only (CLIP text embeddings → FAISS)
print('Mode C — Text only (CLIP text search)')
results = search_multimodal(
    None,
    text_query='navy geometric modern rug',
    top_k=5,
    csv_path=DATA_PATH,
)
for r in results:
    print(f"  #{r['rank']} {r['title'][:60]:60s} text_sim={r['text_similarity']:.3f}")

---
## Part 1 vs Part 2 Comparison

For the same query, compare which model surfaces better results.

In [ ]:
query = 'grey modern geometric rug'
room_img = str(IMAGES_DIR / 'room4.jpg')

print(f'Query: "{query}"\n')

print('=== Part 1 (SentenceTransformer) ===')
for r in search_part1(query, csv_path=DATA_PATH, top_k=5):
    print(f"  #{r['rank']} {r['title'][:60]:60s} {r['final_score']:.3f}")

print('\n=== Part 2 (CLIP + FAISS) ===')
for r in search_multimodal(room_img, text_query=query, top_k=5, csv_path=DATA_PATH):
    print(f"  #{r['rank']} {r['title'][:60]:60s} {r['fusion_score']:.3f}")